# 🔬 CDLib — Colab Training Driver

**Zero model/loss/data logic here.** This notebook only:
1. Bootstraps the environment (clone, install)
2. Links results to Google Drive (survives restarts)
3. Runs training via CLI
4. Plots results

All logic lives in the `cdlib` package.

## Cell 1: Bootstrap

In [ ]:
# Clone repo and install
!git clone https://github.com/Deep-NeuralNetworks-Research-Project/baseline-infa-bula.git cdlib-repo 2>/dev/null || (cd cdlib-repo && git pull)
%cd cdlib-repo
!pip install -e . -q

## Cell 2: Mount Drive + Link Results (survives restarts)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project directory on Drive if it doesn't exist
!mkdir -p /content/drive/MyDrive/cd-project/results
!mkdir -p /content/drive/MyDrive/cd-project/checkpoints

# Symlink so results/ and checkpoints/ persist on Drive
!ln -sf /content/drive/MyDrive/cd-project/results results
!ln -sf /content/drive/MyDrive/cd-project/checkpoints checkpoints

print('✅ Drive mounted. Results will persist across restarts.')

## Cell 3: W&B Login (via Colab Secrets)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('✅ W&B key loaded from Colab Secrets.')
except Exception:
    print('⚠️  W&B key not found in Secrets. Add it via the 🔑 icon in the sidebar.')
    print('   Training will continue without W&B logging.')

## Cell 4: Train (one command)

In [ ]:
# Change the experiment config as needed:
#   +experiment=baseline_fcsiamdiff_sysu
#   +experiment=ablation_no_alignment
#   resume_from=auto  (to continue after a restart)
#
# To run a full baseline sweep:
# !python -m cdlib.cli.train -m train.optimizer.lr=1e-3,3e-4,1e-4 model=fc_siam_diff,rgb_ssim data=sysu_cd,levir_cd train.seed=0,1,2

!python -m cdlib.cli.train +experiment=baseline_fcsiamdiff_sysu

## Cell 5: Resume after restart

In [ ]:
# If Colab disconnected, re-run cells 1-3, then run this:
# !python -m cdlib.cli.train +experiment=baseline_fcsiamdiff_sysu train.checkpoint.resume_from=auto#
# To run a full baseline sweep:
# !python -m cdlib.cli.train -m train.optimizer.lr=1e-3,3e-4,1e-4 model=fc_siam_diff,rgb_ssim data=sysu_cd,levir_cd train.seed=0,1,2


## Cell 6: Plot Results

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# Find the latest run directory
results_dir = Path('results')
if results_dir.exists():
    runs = sorted(results_dir.iterdir())
    if runs:
        latest_run = runs[-1]
        print(f'Latest run: {latest_run}')
        print(f'Contents: {list(latest_run.iterdir())}')
    else:
        print('No runs found yet.')
else:
    print('results/ directory not found. Run training first.')